# Newsgroup Document Classification — Mistral 7B + LoRA (CUDA)

Two-phase fine-tuning pipeline for 20 Newsgroups classification using Mistral 7B with LoRA adapters.

- **Phase 1**: 5 well-separated categories (3 epochs) — validates the pipeline
- **Phase 2**: All 20 categories (5 epochs) — full classification challenge

Requires: NVIDIA GPU with >= 24 GB VRAM (e.g., A100, A6000, RTX 4090)

## 1. Setup & Dependencies

In [ ]:
!pip install torch>=2.1.0 transformers>=4.36.0 accelerate>=0.25.0 peft>=0.8.0 \
    safetensors>=0.4.0 scikit-learn>=1.3.1 seaborn>=0.12.0 matplotlib>=3.7.0 numpy>=1.24.0

In [ ]:
import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    print("WARNING: No CUDA GPU detected. Training will not work.")

## 2. Configuration

Writes `config.py` to disk so training/evaluation code can `import config`. Edit values here and re-run the cell to update.

In [ ]:
%%writefile config.py
"""
CUDA Configuration — Centralized hyperparameters and paths.

All magic numbers live here. Training, evaluation, and inference scripts
import from this module so there's a single source of truth.
"""

from pathlib import Path

# ---------------------------------------------------------------------------
# Model
# ---------------------------------------------------------------------------
MODEL_ID = "mistralai/Mistral-7B-v0.1"
HIDDEN_SIZE = 4096  # Mistral 7B's hidden dimension

# ---------------------------------------------------------------------------
# LoRA — applied to attention Q and V projections across all 32 layers
# ---------------------------------------------------------------------------
LORA_RANK = 16           # Low-rank dimension; 16 balances capacity vs. overfitting for 20 classes
LORA_ALPHA = 16          # Scaling = alpha/rank = 1.0 — no amplification of adapter updates
LORA_DROPOUT = 0.05      # Light regularization; chunking already provides data augmentation
LORA_TARGET_MODULES = ["q_proj", "v_proj"]
LORA_BIAS = "none"       # No bias adaptation — keeps trainable param count minimal

# ---------------------------------------------------------------------------
# Classification Head
# ---------------------------------------------------------------------------
CLASSIFIER_DROPOUT = 0.1  # Between last-token hidden state and linear head

# ---------------------------------------------------------------------------
# Training — Phase 1 (5 well-separated categories)
# ---------------------------------------------------------------------------
PHASE1_EPOCHS = 3
PHASE1_NUM_CLASSES = 5

# ---------------------------------------------------------------------------
# Training — Phase 2 (all 20 categories)
# ---------------------------------------------------------------------------
PHASE2_EPOCHS = 5
PHASE2_NUM_CLASSES = 20

# ---------------------------------------------------------------------------
# Training — Shared across phases
# ---------------------------------------------------------------------------
LEARNING_RATE = 1e-4      # Validated in reference notebook for Mistral 7B + LoRA
WEIGHT_DECAY = 0.01       # Standard L2 regularization for AdamW
MIN_LR = 1e-6             # Cosine scheduler floor — prevents LR from reaching zero
BATCH_SIZE = 4            # Fits within 24 GB VRAM with float16 model + activations
GRAD_ACCUMULATION_STEPS = 4  # Effective batch size = 4 * 4 = 16
EVAL_FREQ = 50            # Validate every N optimizer steps
EVAL_BATCHES = 5          # Number of batches for mid-training eval (speed vs. accuracy)

# ---------------------------------------------------------------------------
# Data
# ---------------------------------------------------------------------------
CHUNK_SIZE = 512
STRIDE = 256
DATA_DIR = Path("data")
PHASE1_DATA = DATA_DIR / "phase1_data.json"
PHASE2_DATA = DATA_DIR / "phase2_data.json"

# ---------------------------------------------------------------------------
# Checkpoints
# ---------------------------------------------------------------------------
CHECKPOINT_DIR = Path("checkpoints")
PHASE1_CHECKPOINT = CHECKPOINT_DIR / "phase1"
PHASE2_CHECKPOINT = CHECKPOINT_DIR / "phase2"

# ---------------------------------------------------------------------------
# Reproducibility
# ---------------------------------------------------------------------------
RANDOM_SEED = 42

## 3. Model Definition

Writes `model.py` to disk. The classification wrapper extracts the last non-padding token's hidden state and projects it through a linear head.

In [ ]:
%%writefile model.py
"""
MistralForSequenceClassification — PyTorch classification wrapper.

This module wraps a PEFT-adapted Mistral model with a classification head.
The key insight: causal LMs process tokens left-to-right, so the *last*
non-padding token has attended to the entire input and holds the richest
representation. We extract that hidden state and project it through a
linear layer to produce class logits.

Also provides load_trained_model() — the single source of truth for
reconstructing a trained model from a checkpoint. Used by evaluate.py
and inference.py.

Used by: train.py, evaluate.py, inference.py
"""

import logging
from pathlib import Path

import safetensors.torch
import torch
import torch.nn as nn
from peft import LoraConfig, TaskType, get_peft_model, set_peft_model_state_dict
from transformers import AutoModelForCausalLM

import config

log = logging.getLogger(__name__)


class MistralForSequenceClassification(nn.Module):
    """Wraps a PEFT Mistral model with dropout + linear classification head.

    Architecture:
        input_ids -> Mistral (frozen + LoRA) -> hidden_states
        -> extract last non-padding token -> dropout -> Linear(4096, num_classes)
        -> logits

    The last-token extraction uses attention_mask to find the correct position,
    handling variable-length sequences within a dynamically-padded batch.
    """

    def __init__(self, base_model: nn.Module, num_labels: int, dropout_rate: float = 0.1):
        super().__init__()
        self.base_model = base_model
        self.num_labels = num_labels
        self.hidden_size = base_model.config.hidden_size

        self.dropout = nn.Dropout(dropout_rate)
        self.classifier = nn.Linear(self.hidden_size, num_labels)

        # Instantiate once in __init__ rather than per forward pass — CE loss
        # has no learnable parameters, but re-creating it each call is wasteful
        self.loss_fn = nn.CrossEntropyLoss()

        # Small std initialization keeps initial logits near zero, so the
        # model starts with roughly uniform class probabilities
        nn.init.normal_(self.classifier.weight, std=0.02)
        nn.init.zeros_(self.classifier.bias)

        # Move the classification head to the same device as the base model.
        # The base model is on GPU via device_map="auto", but nn.Linear
        # defaults to CPU — without this, the forward pass would crash with
        # a device mismatch error when the classifier receives GPU tensors.
        device = next(base_model.parameters()).device
        self.classifier = self.classifier.to(device)

    def forward(
        self,
        input_ids: torch.Tensor,
        attention_mask: torch.Tensor | None = None,
        labels: torch.Tensor | None = None,
    ) -> dict[str, torch.Tensor | None]:
        """Forward pass: base model -> last-token extraction -> classification.

        Args:
            input_ids: (batch_size, seq_len) token IDs.
            attention_mask: (batch_size, seq_len) binary mask, 1 for real tokens.
            labels: (batch_size,) integer class labels. If provided, computes loss.

        Returns:
            Dict with 'loss' (if labels given) and 'logits' (batch_size, num_classes).
        """
        # output_hidden_states=True gives us access to the final layer's
        # hidden states, which we need for classification
        outputs = self.base_model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=True,
        )

        # Shape: (batch_size, seq_len, hidden_size)
        hidden_states = outputs.hidden_states[-1]

        if attention_mask is not None:
            # For each sequence, find the index of the last real (non-padding)
            # token. attention_mask.sum(dim=1) gives the count of real tokens,
            # subtract 1 for 0-based indexing.
            # Example: mask=[1,1,1,0,0] -> sum=3 -> last_token_idx=2
            sequence_lengths = attention_mask.sum(dim=1) - 1
            batch_indices = torch.arange(
                hidden_states.shape[0], device=hidden_states.device
            )
            last_hidden = hidden_states[batch_indices, sequence_lengths]
        else:
            # No padding — just use the final position
            last_hidden = hidden_states[:, -1, :]

        last_hidden = self.dropout(last_hidden)
        logits = self.classifier(last_hidden)

        loss = None
        if labels is not None:
            loss = self.loss_fn(logits, labels)

        return {"loss": loss, "logits": logits}


# ---------------------------------------------------------------------------
# Checkpoint Loading — single source of truth for evaluate.py & inference.py
# ---------------------------------------------------------------------------

def load_trained_model(
    num_classes: int,
    checkpoint_dir: Path,
    device: str,
) -> MistralForSequenceClassification:
    """Reconstruct the model and load saved LoRA adapters + classification head."""
    log.info(f"Loading base model: {config.MODEL_ID}")
    base_model = AutoModelForCausalLM.from_pretrained(
        config.MODEL_ID,
        torch_dtype=torch.float16,
        device_map="auto",
    )

    lora_config = LoraConfig(
        r=config.LORA_RANK,
        lora_alpha=config.LORA_ALPHA,
        target_modules=config.LORA_TARGET_MODULES,
        lora_dropout=config.LORA_DROPOUT,
        bias=config.LORA_BIAS,
        task_type=TaskType.FEATURE_EXTRACTION,
    )
    base_model = get_peft_model(base_model, lora_config)

    lora_path = checkpoint_dir / "lora_adapters"
    log.info(f"Loading LoRA adapters from {lora_path}")
    adapter_weights = safetensors.torch.load_file(
        str(lora_path / "adapter_model.safetensors")
    )
    set_peft_model_state_dict(base_model, adapter_weights)

    model = MistralForSequenceClassification(
        base_model, num_labels=num_classes, dropout_rate=config.CLASSIFIER_DROPOUT,
    )

    head_path = checkpoint_dir / "classifier_head.pt"
    log.info(f"Loading classification head from {head_path}")
    head_state = torch.load(head_path, map_location="cpu", weights_only=True)
    model.classifier.weight.data = head_state["classifier.weight"].to(device)
    model.classifier.bias.data = head_state["classifier.bias"].to(device)

    model.eval()
    return model

## 4. Data Preparation

Downloads 20 Newsgroups, tokenizes with Mistral's tokenizer, chunks long documents, and saves JSON files for both phases.

In [ ]:
import json
import re
import time
from pathlib import Path

import numpy as np
from sklearn.datasets import fetch_20newsgroups
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer

# --- Configuration ---
MODEL_ID = "mistralai/Mistral-7B-v0.1"
CHUNK_SIZE = 512
STRIDE = 256
VAL_SPLIT = 0.15
RANDOM_SEED = 42
OUTPUT_DIR = Path("data")

PHASE1_CATEGORIES = [
    "rec.sport.hockey",
    "sci.space",
    "comp.graphics",
    "talk.politics.mideast",
    "soc.religion.christian",
]


def clean_text(text: str) -> str:
    text = text.strip()
    if not text:
        return "[empty post]"
    return re.sub(r"\n{3,}", "\n\n", text)


def chunk_tokens(tokens: list[int], chunk_size: int, stride: int) -> list[list[int]]:
    if len(tokens) <= chunk_size:
        return [tokens]
    chunks = []
    for start in range(0, len(tokens), stride):
        chunk = tokens[start : start + chunk_size]
        chunks.append(chunk)
        if start + chunk_size >= len(tokens):
            break
    return chunks


def process_split(texts, labels, tokenizer, chunk_size, stride, split_name, track_doc_ids=False):
    all_chunks, all_labels = [], []
    all_doc_ids = [] if track_doc_ids else None
    chunked_count = 0

    for doc_id, (text, label) in enumerate(zip(texts, labels)):
        text = clean_text(text)
        tokens = tokenizer.encode(text, add_special_tokens=False)
        doc_chunks = chunk_tokens(tokens, chunk_size, stride)
        if len(doc_chunks) > 1:
            chunked_count += 1
        for chunk in doc_chunks:
            all_chunks.append(chunk)
            all_labels.append(int(label))
            if track_doc_ids:
                all_doc_ids.append(doc_id)

    chunk_lengths = [len(c) for c in all_chunks]
    print(f"  {split_name}: {len(texts)} docs -> {len(all_chunks)} chunks")
    print(f"    {chunked_count} docs ({100 * chunked_count / len(texts):.1f}%) split into multiple chunks")
    print(f"    Chunk lengths — min: {min(chunk_lengths)}, max: {max(chunk_lengths)}, "
          f"mean: {np.mean(chunk_lengths):.0f}, median: {np.median(chunk_lengths):.0f}")

    result = {"chunks": all_chunks, "labels": all_labels}
    if track_doc_ids:
        result["doc_ids"] = all_doc_ids
    return result


def prepare_phase(categories, tokenizer, phase_name, output_path):
    print(f"\n{'=' * 60}")
    print(f"Preparing {phase_name}")
    print(f"{'=' * 60}")

    print("Fetching data from sklearn (remove headers/footers/quotes)...")
    train_raw = fetch_20newsgroups(
        subset="train", categories=categories,
        remove=("headers", "footers", "quotes"), random_state=RANDOM_SEED,
    )
    test_raw = fetch_20newsgroups(
        subset="test", categories=categories,
        remove=("headers", "footers", "quotes"), random_state=RANDOM_SEED,
    )

    label_names = list(train_raw.target_names)
    num_classes = len(label_names)
    print(f"Categories ({num_classes}): {label_names}")
    print(f"Raw train: {len(train_raw.data)}, Raw test: {len(test_raw.data)}")

    train_texts, val_texts, train_labels, val_labels = train_test_split(
        train_raw.data, train_raw.target,
        test_size=VAL_SPLIT, random_state=RANDOM_SEED, stratify=train_raw.target,
    )
    test_texts, test_labels = test_raw.data, test_raw.target
    print(f"After split — train: {len(train_texts)}, val: {len(val_texts)}, test: {len(test_texts)}")

    print(f"\nTokenizing and chunking (chunk_size={CHUNK_SIZE}, stride={STRIDE})...")
    train_data = process_split(train_texts, train_labels, tokenizer, CHUNK_SIZE, STRIDE, "train")
    val_data = process_split(val_texts, val_labels, tokenizer, CHUNK_SIZE, STRIDE, "val", track_doc_ids=True)
    test_data = process_split(test_texts, test_labels, tokenizer, CHUNK_SIZE, STRIDE, "test", track_doc_ids=True)

    output = {
        "train_chunks": train_data["chunks"], "train_labels": train_data["labels"],
        "val_chunks": val_data["chunks"], "val_labels": val_data["labels"],
        "val_doc_ids": val_data["doc_ids"], "val_num_docs": len(val_texts),
        "val_doc_labels": [int(l) for l in val_labels],
        "test_chunks": test_data["chunks"], "test_labels": test_data["labels"],
        "test_doc_ids": test_data["doc_ids"], "test_num_docs": len(test_texts),
        "test_doc_labels": [int(l) for l in test_labels],
        "label_names": label_names, "num_classes": num_classes,
        "chunk_size": CHUNK_SIZE, "stride": STRIDE,
    }

    output_path.parent.mkdir(parents=True, exist_ok=True)
    with open(output_path, "w") as f:
        json.dump(output, f)

    file_size_mb = output_path.stat().st_size / (1024 * 1024)
    print(f"\nSaved to {output_path} ({file_size_mb:.1f} MB)")


# --- Run data preparation ---
np.random.seed(RANDOM_SEED)
start_time = time.time()

print(f"Loading tokenizer: {MODEL_ID}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

print(f"Vocabulary size: {len(tokenizer):,}")
print(f"Pad token: '{tokenizer.pad_token}' (ID: {tokenizer.pad_token_id})")

prepare_phase(PHASE1_CATEGORIES, tokenizer, "Phase 1 (5 categories)", OUTPUT_DIR / "phase1_data.json")
prepare_phase(None, tokenizer, "Phase 2 (20 categories)", OUTPUT_DIR / "phase2_data.json")

print(f"\nData preparation complete in {time.time() - start_time:.1f}s")

## 5. Training Utilities

Dataset, collation, model building, loss computation, and the core training loop.

In [ ]:
import importlib
import json
import logging
import time
from functools import partial

import numpy as np
import torch
import torch.nn as nn
from peft import LoraConfig, TaskType, get_peft_model
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

# Reload config in case the %%writefile cell was re-run
import config
importlib.reload(config)
from model import MistralForSequenceClassification

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%H:%M:%S",
    force=True,
)
log = logging.getLogger("train")


def set_seed(seed: int) -> None:
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


class ChunkDataset(Dataset):
    def __init__(self, chunks, labels, doc_ids=None):
        self.chunks = chunks
        self.labels = labels
        self.doc_ids = doc_ids

    def __len__(self):
        return len(self.chunks)

    def __getitem__(self, index):
        item = {"input_ids": self.chunks[index], "labels": self.labels[index]}
        if self.doc_ids is not None:
            item["doc_id"] = self.doc_ids[index]
        return item


def dynamic_padding_collate(batch, pad_token_id):
    input_ids_list = [item["input_ids"] for item in batch]
    labels = torch.tensor([item["labels"] for item in batch], dtype=torch.long)
    max_len = max(len(ids) for ids in input_ids_list)

    padded_ids, attention_masks = [], []
    for ids in input_ids_list:
        pad_len = max_len - len(ids)
        padded_ids.append(ids + [pad_token_id] * pad_len)
        attention_masks.append([1] * len(ids) + [0] * pad_len)

    result = {
        "input_ids": torch.tensor(padded_ids, dtype=torch.long),
        "attention_mask": torch.tensor(attention_masks, dtype=torch.long),
        "labels": labels,
    }
    if "doc_id" in batch[0]:
        result["doc_id"] = torch.tensor([item["doc_id"] for item in batch], dtype=torch.long)
    return result


def load_phase_data(data_path):
    log.info(f"Loading data from {data_path}")
    with open(data_path, "r") as f:
        data = json.load(f)
    log.info(f"  Classes: {data['num_classes']}, "
             f"Train chunks: {len(data['train_chunks'])}, "
             f"Val chunks: {len(data['val_chunks'])}, "
             f"Test chunks: {len(data['test_chunks'])}")
    return data


def create_dataloaders(data, pad_token_id):
    train_dataset = ChunkDataset(data["train_chunks"], data["train_labels"])
    val_dataset = ChunkDataset(data["val_chunks"], data["val_labels"], doc_ids=data["val_doc_ids"])
    test_dataset = ChunkDataset(data["test_chunks"], data["test_labels"], doc_ids=data["test_doc_ids"])

    collate_fn = partial(dynamic_padding_collate, pad_token_id=pad_token_id)

    train_loader = DataLoader(train_dataset, batch_size=config.BATCH_SIZE, shuffle=True,
                              num_workers=0, drop_last=True, collate_fn=collate_fn)
    val_loader = DataLoader(val_dataset, batch_size=config.BATCH_SIZE, shuffle=False,
                            num_workers=0, drop_last=False, collate_fn=collate_fn)
    test_loader = DataLoader(test_dataset, batch_size=config.BATCH_SIZE, shuffle=False,
                             num_workers=0, drop_last=False, collate_fn=collate_fn)
    return train_loader, val_loader, test_loader


def build_model(num_classes):
    log.info(f"Loading tokenizer: {config.MODEL_ID}")
    tokenizer = AutoTokenizer.from_pretrained(config.MODEL_ID)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
        tokenizer.pad_token_id = tokenizer.eos_token_id

    log.info("Loading base model in float16 (this takes 1-2 minutes)...")
    base_model = AutoModelForCausalLM.from_pretrained(
        config.MODEL_ID, torch_dtype=torch.float16, device_map="auto",
    )
    log.info(f"Model loaded. Hidden size: {base_model.config.hidden_size}, "
             f"Layers: {base_model.config.num_hidden_layers}")

    lora_config = LoraConfig(
        r=config.LORA_RANK, lora_alpha=config.LORA_ALPHA,
        target_modules=config.LORA_TARGET_MODULES, lora_dropout=config.LORA_DROPOUT,
        bias=config.LORA_BIAS, task_type=TaskType.FEATURE_EXTRACTION,
    )
    base_model = get_peft_model(base_model, lora_config)
    base_model.print_trainable_parameters()

    model = MistralForSequenceClassification(
        base_model, num_labels=num_classes, dropout_rate=config.CLASSIFIER_DROPOUT,
    )

    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    log.info(f"Classification model ready:")
    log.info(f"  Total params: {total_params:,}")
    log.info(f"  Trainable params: {trainable_params:,} ({100 * trainable_params / total_params:.3f}%)")
    log.info(f"  Classifier head: Linear({model.hidden_size} -> {num_classes})")

    return model, tokenizer


def compute_loss(data_loader, model, device, num_batches):
    model.eval()
    total_loss = 0.0
    count = min(num_batches, len(data_loader))
    with torch.no_grad():
        for i, batch in enumerate(data_loader):
            if i >= count:
                break
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            total_loss += outputs["loss"].item()
    return total_loss / count


def compute_doc_accuracy(data_loader, model, device, doc_labels, num_batches=None):
    model.eval()
    doc_logits = {}
    with torch.no_grad():
        for i, batch in enumerate(data_loader):
            if num_batches is not None and i >= num_batches:
                break
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            doc_ids = batch["doc_id"].numpy()
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            logits = outputs["logits"].cpu()
            for j, doc_id in enumerate(doc_ids):
                doc_id = int(doc_id)
                if doc_id not in doc_logits:
                    doc_logits[doc_id] = []
                doc_logits[doc_id].append(logits[j])

    correct = 0
    for doc_id, logits_list in doc_logits.items():
        avg_logits = torch.stack(logits_list).mean(dim=0)
        pred = torch.argmax(avg_logits).item()
        if pred == doc_labels[doc_id]:
            correct += 1
    return correct / len(doc_logits) if doc_logits else 0.0


def train_phase(model, train_loader, val_loader, val_doc_labels, num_epochs, checkpoint_dir, phase_name):
    device = "cuda"
    # Only pass trainable parameters (LoRA adapters + classification head) to
    # the optimizer — no need to iterate over ~7.2B frozen base model params.
    trainable_params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(trainable_params, lr=config.LEARNING_RATE, weight_decay=config.WEIGHT_DECAY)

    # T_max must equal the number of *optimizer steps* (not batch steps),
    # because scheduler.step() is called once per optimizer step (after
    # gradient accumulation). Using len(train_loader) directly would set
    # T_max ~4x too high, causing the LR to barely decay over training.
    total_steps = num_epochs * (len(train_loader) // config.GRAD_ACCUMULATION_STEPS)
    scheduler = CosineAnnealingLR(optimizer, T_max=total_steps, eta_min=config.MIN_LR)

    log.info(f"\n{'=' * 60}")
    log.info(f"Starting {phase_name}")
    log.info(f"{'=' * 60}")
    log.info(f"Epochs: {num_epochs}, Batch size: {config.BATCH_SIZE}, "
             f"Grad accum: {config.GRAD_ACCUMULATION_STEPS}, "
             f"Effective batch: {config.BATCH_SIZE * config.GRAD_ACCUMULATION_STEPS}")
    log.info(f"LR: {config.LEARNING_RATE} -> {config.MIN_LR} (cosine), Total steps: {total_steps}")

    train_losses, val_losses, epoch_accs = [], [], []
    global_step = 0
    phase_start = time.time()

    for epoch in range(num_epochs):
        model.train()
        epoch_start = time.time()
        epoch_loss = 0.0
        batch_count = 0

        for batch_idx, batch in enumerate(train_loader):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs["loss"] / config.GRAD_ACCUMULATION_STEPS
            loss.backward()

            if (batch_idx + 1) % config.GRAD_ACCUMULATION_STEPS == 0:
                optimizer.step()
                optimizer.zero_grad()
                scheduler.step()
                global_step += 1

                if global_step % config.EVAL_FREQ == 0:
                    train_loss = compute_loss(train_loader, model, device, config.EVAL_BATCHES)
                    val_loss = compute_loss(val_loader, model, device, config.EVAL_BATCHES)
                    train_losses.append(train_loss)
                    val_losses.append(val_loss)
                    log.info(
                        f"  Epoch {epoch + 1}/{num_epochs} | Step {global_step:05d} | "
                        f"Train loss: {train_loss:.4f} | Val loss: {val_loss:.4f} | "
                        f"LR: {scheduler.get_last_lr()[0]:.2e}"
                    )
                    model.train()

            epoch_loss += outputs["loss"].item()
            batch_count += 1

        val_acc = compute_doc_accuracy(val_loader, model, device, val_doc_labels)
        epoch_accs.append(val_acc)
        epoch_time = time.time() - epoch_start
        log.info(f"  Epoch {epoch + 1} complete — Avg loss: {epoch_loss / batch_count:.4f}, "
                 f"Val accuracy (doc-level): {val_acc * 100:.1f}%, Time: {epoch_time:.0f}s")

    # Save checkpoints
    checkpoint_dir = config.CHECKPOINT_DIR / checkpoint_dir
    checkpoint_dir.mkdir(parents=True, exist_ok=True)

    model.base_model.save_pretrained(str(checkpoint_dir / "lora_adapters"))
    head_state = {
        "classifier.weight": model.classifier.weight.data.cpu(),
        "classifier.bias": model.classifier.bias.data.cpu(),
        "num_labels": model.num_labels,
    }
    torch.save(head_state, str(checkpoint_dir / "classifier_head.pt"))

    phase_time = (time.time() - phase_start) / 60
    log.info(f"\n{phase_name} complete in {phase_time:.1f} minutes")
    log.info(f"Checkpoints saved to {checkpoint_dir}")

    return {"train_losses": train_losses, "val_losses": val_losses,
            "epoch_accs": epoch_accs, "training_time_min": phase_time}


print("Training utilities loaded.")

## 6. Phase 1 — 5-Class Training

In [ ]:
set_seed(config.RANDOM_SEED)

phase1_data = load_phase_data(config.PHASE1_DATA)
model, tokenizer = build_model(num_classes=phase1_data["num_classes"])
train_loader, val_loader, _ = create_dataloaders(phase1_data, tokenizer.pad_token_id)

phase1_results = train_phase(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    val_doc_labels=phase1_data["val_doc_labels"],
    num_epochs=config.PHASE1_EPOCHS,
    checkpoint_dir="phase1",
    phase_name="Phase 1 (5-class)",
)

# Free GPU memory before Phase 2
del model, train_loader, val_loader
torch.cuda.empty_cache()

print(f"\nPhase 1 final val accuracies: {[f'{a*100:.1f}%' for a in phase1_results['epoch_accs']]}")

## 7. Phase 2 — 20-Class Training

In [ ]:
set_seed(config.RANDOM_SEED)

phase2_data = load_phase_data(config.PHASE2_DATA)
model, tokenizer = build_model(num_classes=phase2_data["num_classes"])
train_loader, val_loader, _ = create_dataloaders(phase2_data, tokenizer.pad_token_id)

phase2_results = train_phase(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    val_doc_labels=phase2_data["val_doc_labels"],
    num_epochs=config.PHASE2_EPOCHS,
    checkpoint_dir="phase2",
    phase_name="Phase 2 (20-class)",
)

print(f"\nPhase 2 final val accuracies: {[f'{a*100:.1f}%' for a in phase2_results['epoch_accs']]}")

## 8. Evaluation

Full test-set evaluation with classification report, confusion matrix, and topic-group breakdown.

In [ ]:
import importlib
import json
from functools import partial
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import torch
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from torch.utils.data import DataLoader
from transformers import AutoTokenizer

import config
importlib.reload(config)
from model import MistralForSequenceClassification, load_trained_model


def evaluate_with_aggregation(model, data_loader, doc_labels, num_docs, label_names, device):
    """Document-level evaluation with logit aggregation across chunks."""
    model.eval()
    doc_logits = {}

    print("Running inference on all chunks...")
    with torch.no_grad():
        for batch in data_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            doc_ids = batch["doc_id"].numpy()
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            logits = outputs["logits"].cpu()
            for i, doc_id in enumerate(doc_ids):
                doc_id = int(doc_id)
                if doc_id not in doc_logits:
                    doc_logits[doc_id] = []
                doc_logits[doc_id].append(logits[i])

    all_preds = np.zeros(num_docs, dtype=np.int64)
    for doc_id in range(num_docs):
        if doc_id in doc_logits:
            avg_logits = torch.stack(doc_logits[doc_id]).mean(dim=0)
            all_preds[doc_id] = torch.argmax(avg_logits).item()

    all_labels = np.array(doc_labels)
    accuracy = (all_preds == all_labels).mean()
    macro_f1 = f1_score(all_labels, all_preds, average="macro")
    weighted_f1 = f1_score(all_labels, all_preds, average="weighted")

    print(f"\n{'=' * 70}")
    print("Classification Report")
    print("=" * 70)
    print(classification_report(all_labels, all_preds, target_names=label_names, digits=3))
    print(f"Document-level Accuracy: {accuracy:.4f}")
    print(f"Macro F1:                {macro_f1:.4f}")
    print(f"Weighted F1:             {weighted_f1:.4f}")

    return all_preds, all_labels


def plot_confusion_matrix(all_labels, all_preds, label_names, title, save_path=None):
    cm = confusion_matrix(all_labels, all_preds)
    short_names = [name.split(".")[-1] for name in label_names]
    figsize = (8, 6) if len(label_names) <= 10 else (14, 12)
    fig, ax = plt.subplots(figsize=figsize)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=short_names, yticklabels=short_names, ax=ax)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_title(title)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150)
        print(f"Confusion matrix saved to {save_path}")
    plt.show()


def print_accuracy_by_group(all_labels, all_preds, label_names):
    groups = {}
    for i, name in enumerate(label_names):
        group = name.split(".")[0]
        if group not in groups:
            groups[group] = {"correct": 0, "total": 0}
        mask = all_labels == i
        groups[group]["total"] += int(mask.sum())
        groups[group]["correct"] += int((all_preds[mask] == i).sum())

    print("\nAccuracy by Topic Group:")
    print(f"{'Group':<10} {'Accuracy':>10} {'Correct/Total':>15}")
    print("-" * 40)
    for group, stats in sorted(groups.items()):
        acc = stats["correct"] / stats["total"] * 100 if stats["total"] > 0 else 0
        print(f"{group:<10} {acc:>9.1f}% {stats['correct']:>6}/{stats['total']:<6}")


def print_top_confused_pairs(all_labels, all_preds, label_names, top_k=10):
    cm = confusion_matrix(all_labels, all_preds)
    cm_off_diag = cm.copy()
    np.fill_diagonal(cm_off_diag, 0)
    pairs = []
    for i in range(len(label_names)):
        for j in range(len(label_names)):
            if i != j and cm_off_diag[i][j] > 0:
                pairs.append((label_names[i], label_names[j], cm_off_diag[i][j]))
    pairs.sort(key=lambda x: x[2], reverse=True)

    print(f"\nTop {top_k} Most Confused Category Pairs:")
    print(f"{'True Category':<35} {'Predicted As':<35} {'Count':>5}")
    print("-" * 80)
    for true_cat, pred_cat, count in pairs[:top_k]:
        print(f"{true_cat:<35} {pred_cat:<35} {count:>5}")


print("Evaluation utilities loaded.")

### Evaluate Phase 2 (20-class)

Change `EVAL_PHASE` to `1` to evaluate Phase 1 instead.

In [ ]:
EVAL_PHASE = 2  # Change to 1 for Phase 1 evaluation

device = "cuda" if torch.cuda.is_available() else "cpu"

if EVAL_PHASE == 1:
    data_path = config.PHASE1_DATA
    checkpoint_dir = config.CHECKPOINT_DIR / "phase1"
    phase_name = "Phase 1 (5-class)"
else:
    data_path = config.PHASE2_DATA
    checkpoint_dir = config.CHECKPOINT_DIR / "phase2"
    phase_name = "Phase 2 (20-class)"

with open(data_path, "r") as f:
    eval_data = json.load(f)

label_names = eval_data["label_names"]
num_classes = eval_data["num_classes"]

eval_tokenizer = AutoTokenizer.from_pretrained(config.MODEL_ID)
if eval_tokenizer.pad_token is None:
    eval_tokenizer.pad_token = eval_tokenizer.eos_token
    eval_tokenizer.pad_token_id = eval_tokenizer.eos_token_id

test_dataset = ChunkDataset(eval_data["test_chunks"], eval_data["test_labels"], doc_ids=eval_data["test_doc_ids"])
collate_fn = partial(dynamic_padding_collate, pad_token_id=eval_tokenizer.pad_token_id)
test_loader = DataLoader(test_dataset, batch_size=config.BATCH_SIZE, shuffle=False,
                         num_workers=0, drop_last=False, collate_fn=collate_fn)

eval_model = load_trained_model(num_classes, checkpoint_dir, device)

print(f"\n{'=' * 70}")
print(f"{phase_name} — Test Set Evaluation")
print(f"{'=' * 70}")

all_preds, all_labels = evaluate_with_aggregation(
    eval_model, test_loader, eval_data["test_doc_labels"],
    eval_data["test_num_docs"], label_names, device,
)

In [ ]:
# Confusion matrix
cm_save_path = str(checkpoint_dir / "confusion_matrix.png")
plot_confusion_matrix(all_labels, all_preds, label_names,
                      title=f"{phase_name} — Confusion Matrix", save_path=cm_save_path)

# Group-level accuracy and confusion analysis (20-class only)
if num_classes > 5:
    print_accuracy_by_group(all_labels, all_preds, label_names)
    print_top_confused_pairs(all_labels, all_preds, label_names)

## 9. Inference Demo

Classify sample texts using the trained model.

In [ ]:
import re

import numpy as np
import torch


def clean_text_inf(text):
    text = text.strip()
    if not text:
        return "[empty post]"
    return re.sub(r"\n{3,}", "\n\n", text)


def chunk_tokens_inf(tokens, chunk_size, stride):
    if len(tokens) <= chunk_size:
        return [tokens]
    chunks = []
    for start in range(0, len(tokens), stride):
        chunk = tokens[start : start + chunk_size]
        chunks.append(chunk)
        if start + chunk_size >= len(tokens):
            break
    return chunks


def classify_document(text, model, tokenizer, device, label_names,
                      chunk_size=config.CHUNK_SIZE, stride=config.STRIDE):
    """Classify a single document with chunk aggregation."""
    model.eval()
    text = clean_text_inf(text)
    tokens = tokenizer.encode(text, add_special_tokens=False)
    chunks = chunk_tokens_inf(tokens, chunk_size, stride)

    all_logits = []
    for chunk in chunks:
        pad_len = chunk_size - len(chunk)
        padded = chunk + [tokenizer.pad_token_id] * pad_len
        mask = [1] * len(chunk) + [0] * pad_len

        input_ids = torch.tensor([padded], dtype=torch.long).to(device)
        attention_mask = torch.tensor([mask], dtype=torch.long).to(device)

        with torch.no_grad():
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            all_logits.append(outputs["logits"].cpu())

    avg_logits = torch.cat(all_logits, dim=0).mean(dim=0)
    probs = torch.softmax(avg_logits, dim=-1).numpy()
    pred_idx = int(np.argmax(probs))

    top3_idx = np.argsort(probs)[-3:][::-1]
    top3 = [(label_names[i], float(probs[i] * 100)) for i in top3_idx]

    return {
        "prediction": label_names[pred_idx],
        "confidence": float(probs[pred_idx] * 100),
        "top3": top3,
        "num_chunks": len(chunks),
        "num_tokens": len(tokens),
    }


# --- Demo texts ---
DEMO_TEXTS = [
    "The Hubble telescope captured amazing images of a distant galaxy cluster.",
    "The Penguins dominated the third period with two quick goals.",
    "How do I render 3D graphics using OpenGL on Linux?",
    "The situation in the Middle East continues to escalate with new tensions.",
    "Jesus taught his disciples the importance of forgiveness and love.",
    "I'm selling my old 486 PC with 8MB RAM and a 200MB hard drive.",
    "The new encryption standard should provide better security for communications.",
    "My motorcycle needs new brake pads. Any recommendations for a Honda CB750?",
]

# Use the eval_model from the previous section (or reload if needed)
inf_model = eval_model
inf_tokenizer = eval_tokenizer

print(f"{'=' * 70}")
print(f"Demo: {phase_name} Document Classification")
print(f"{'=' * 70}\n")

for text in DEMO_TEXTS:
    result = classify_document(text, inf_model, inf_tokenizer, device, label_names)
    display_text = text[:70] + "..." if len(text) > 70 else text
    print(f"Text: {display_text}")
    print(f"  Prediction: {result['prediction']} ({result['confidence']:.1f}%)")
    print(f"  Top 3: {', '.join(f'{n} ({c:.1f}%)' for n, c in result['top3'])}")
    print(f"  [{result['num_tokens']} tokens, {result['num_chunks']} chunk(s)]")
    print()

### Try your own text

In [ ]:
custom_text = "Enter your own text here to classify it."

result = classify_document(custom_text, inf_model, inf_tokenizer, device, label_names)
print(f"Prediction: {result['prediction']} ({result['confidence']:.1f}%)")
print(f"Top 3: {', '.join(f'{n} ({c:.1f}%)' for n, c in result['top3'])}")
print(f"[{result['num_tokens']} tokens, {result['num_chunks']} chunk(s)]")